# 2 - Pre-processamento v2

Esta versao 2 continua mais forte que a versao 1, mas agora volta para um regime de **arquivo unico**. A ideia e acelerar a busca pelo modelo ideal: primeiro ajustamos a arquitetura em um caso controlado; depois, em uma versao futura, expandimos para muitos arquivos.


## O que muda nesta adaptacao

Em vez de misturar varias series do dataset, o fluxo fica assim:

- escolhe um unico arquivo normal do 3W
- limpa esse arquivo
- divide a serie temporal em treino, validacao e teste
- gera features derivadas mais informativas
- salva os artefatos da versao 2 para treino e avaliacao

Isso preserva o desenho metodologico mais forte da versao 2, mas reduz custo e complexidade durante a fase de prototipacao.


In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from pipeline_v2 import *

#configura reproducibilidade e exibicao
set_seed(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


## 1. Configuracoes

Por padrao, usamos o mesmo arquivo de referencia da versao 1. Se quiser trocar de evento depois, basta alterar `SOURCE_FILE`.


In [2]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in {"versao1", "versao2"} else CURRENT_DIR
DATASET_ROOT = PROJECT_ROOT / "3W" / "dataset"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
PREPROCESSED_V2_DIR = ARTIFACTS_DIR / "preprocessed_v2"
BUNDLE_PATH = PREPROCESSED_V2_DIR / "preprocessing_bundle_v2.json"

PREFERRED_SOURCE_FILE = DATASET_ROOT / "0" / "WELL-00001_20170201010207.parquet"
if PREFERRED_SOURCE_FILE.exists():
    SOURCE_FILE = PREFERRED_SOURCE_FILE
else:
    candidate_files = sorted((DATASET_ROOT / "0").glob("*.parquet"))
    if not candidate_files:
        raise FileNotFoundError("Nenhum arquivo .parquet foi encontrado em 3W/dataset/0.")
    SOURCE_FILE = candidate_files[0]

ROLLING_WINDOW = 5
SEQUENCE_LENGTH = 60

PREPROCESSED_V2_DIR.mkdir(parents=True, exist_ok=True)
for split_name in ["train", "validation", "test"]:
    split_dir = PREPROCESSED_V2_DIR / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    for old_file in split_dir.glob("*.parquet"):
        old_file.unlink()

{
    "source_file": str(SOURCE_FILE),
    "rolling_window": ROLLING_WINDOW,
    "sequence_length": SEQUENCE_LENGTH,
}


{'source_file': '/home/tiagoriosrocha/Desktop/lstm-w3/3W/dataset/0/WELL-00001_20170201010207.parquet',
 'rolling_window': 5,
 'sequence_length': 60}

## 2. Leitura e limpeza base

Aqui mantemos a filosofia da versao 2:

- limpar com cuidado os alvos
- tratar sinais de estado separadamente
- permitir variaveis auxiliares apenas quando elas realmente variam


In [3]:
#carrega e limpa o arquivo unico da versao2
clean_df = clean_base_frame(SOURCE_FILE)

base_summary_df = pd.DataFrame(
    {
        "aspecto": ["linhas", "colunas", "timestamp_inicial", "timestamp_final"],
        "valor": [
            len(clean_df),
            len(clean_df.columns),
            clean_df["timestamp"].min(),
            clean_df["timestamp"].max(),
        ],
    }
)

display(base_summary_df)
clean_df.head()


,aspecto,valor
0,linhas,21474
1,colunas,19
2,timestamp_inicial,2017-02-01 01:02:07
3,timestamp_final,2017-02-01 07:00:00


,timestamp,P-ANULAR,P-JUS-CKGL,P-MON-CKP,P-TPT,T-JUS-CKP,T-TPT,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,ESTADO-XO,P-PDG,QGL,T-PDG
0,2017-02-01 01:02:07,12767730.0,1563422.0,1627884.0,10074540.0,84.64463,119.0781,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
1,2017-02-01 01:02:08,12767730.0,1563422.0,1633397.0,10074540.0,84.63828,119.0781,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
2,2017-02-01 01:02:09,12767730.0,1563422.0,1638909.0,10074540.0,84.63194,119.0781,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
3,2017-02-01 01:02:10,12767730.0,1563422.0,1644422.0,10074540.0,84.62558,119.0781,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
4,2017-02-01 01:02:11,12767730.0,1563423.0,1649934.0,10074540.0,84.61923,119.0781,1.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0


In [4]:
#faz o split temporal dentro do mesmo arquivo
split_frames = split_single_frame_temporally(clean_df, train_frac=0.7, validation_frac=0.15)
train_df = split_frames["train"]
validation_df = split_frames["validation"]
test_df = split_frames["test"]

split_overview_df = pd.DataFrame(
    [
        {"split": "train", "rows": len(train_df), "start": train_df["timestamp"].min(), "end": train_df["timestamp"].max()},
        {"split": "validation", "rows": len(validation_df), "start": validation_df["timestamp"].min(), "end": validation_df["timestamp"].max()},
        {"split": "test", "rows": len(test_df), "start": test_df["timestamp"].min(), "end": test_df["timestamp"].max()},
    ]
)
split_overview_df


,split,rows,start,end
0,train,15031,2017-02-01 01:02:07,2017-02-01 05:12:37
1,validation,3221,2017-02-01 05:12:38,2017-02-01 06:06:18
2,test,3222,2017-02-01 06:06:19,2017-02-01 07:00:00


## 3. Escolha das variaveis auxiliares

As 6 variaveis alvo continuam as mesmas, mas a versao 2 ainda tenta enriquecer a entrada com variaveis auxiliares. Essa selecao usa apenas o conjunto de treino para evitar vazamento.


In [5]:
#mede variabilidade das colunas auxiliares usando apenas o treino
auxiliary_reference_df = train_df[BASE_TARGET_COLUMNS + [col for col in STATE_COLUMNS + AUX_ANALOG_COLUMNS if col in train_df.columns]].copy()
auxiliary_summary_df = select_auxiliary_columns(auxiliary_reference_df)
auxiliary_summary_df


,column,null_pct,nunique,std,selected_for_input
0,ESTADO-DHSV,0.0,1,0.0,False
1,ESTADO-M1,0.0,1,0.0,False
2,ESTADO-M2,0.0,1,0.0,False
3,ESTADO-PXO,0.0,1,0.0,False
4,ESTADO-SDV-GL,0.0,1,0.0,False
5,ESTADO-SDV-P,0.0,1,0.0,False
6,ESTADO-W1,0.0,1,0.0,False
7,ESTADO-W2,0.0,1,0.0,False
8,ESTADO-XO,0.0,1,0.0,False
9,P-PDG,0.0,1,0.0,False


In [6]:
#ajusta os scalers da versao2 usando somente o trecho de treino
selected_auxiliary_columns = auxiliary_summary_df.loc[
    auxiliary_summary_df["selected_for_input"],
    "column",
].tolist()

well_name = SOURCE_FILE.stem.split("_")[0]

bundle = fit_preprocessing_bundle_from_train_frame(
    train_frame=train_df,
    auxiliary_columns=selected_auxiliary_columns,
    well_name=well_name,
    source_file=str(SOURCE_FILE.resolve()),
    rolling_window=ROLLING_WINDOW,
    sequence_length_recommendation=SEQUENCE_LENGTH,
)
save_bundle(bundle, BUNDLE_PATH)

{
    "target_columns": bundle.target_columns,
    "auxiliary_columns": bundle.auxiliary_columns,
    "n_input_features": len(bundle.input_columns),
}


{'target_columns': ['P-ANULAR',
  'P-JUS-CKGL',
  'P-MON-CKP',
  'P-TPT',
  'T-JUS-CKP',
  'T-TPT'],
 'auxiliary_columns': [],
 'n_input_features': 24}

## 4. Geracao das features de entrada

Mesmo com um unico arquivo, mantemos o diferencial da versao 2:

- `raw__`: sinal padronizado
- `diff1__`: velocidade local
- `dev_roll5__`: afastamento da media curta
- `std_roll5__`: volatilidade local

Isso da ao modelo mais contexto do que uma LSTM simples alimentada so pelo valor bruto.


In [7]:
#transforma cada split em um parquet pronto para modelagem
series_base_name = SOURCE_FILE.stem

transformed_splits = {}
for split_name, split_df in split_frames.items():
    engineered_df = transform_clean_frame_to_engineered_features(
        frame=split_df,
        bundle=bundle,
        series_id=f"{split_name}__{series_base_name}",
        well_name=well_name,
        file_path=str(SOURCE_FILE.resolve()),
    )
    output_path = PREPROCESSED_V2_DIR / split_name / f"{split_name}__{series_base_name}.parquet"
    engineered_df.to_parquet(output_path, index=False)
    transformed_splits[split_name] = engineered_df

transformed_split_summary_df = pd.DataFrame(
    [
        {
            "split": split_name,
            "rows": len(split_df),
            "input_features": len(bundle.input_columns),
            "target_features": len(bundle.target_columns),
        }
        for split_name, split_df in transformed_splits.items()
    ]
)
transformed_split_summary_df


,split,rows,input_features,target_features
0,train,15031,24,6
1,validation,3221,24,6
2,test,3222,24,6


In [8]:
#abre uma serie ja transformada para inspecao didatica
sample_files = sorted((PREPROCESSED_V2_DIR / "train").glob("*.parquet"))
if not sample_files:
    raise FileNotFoundError("Nenhum arquivo .parquet foi encontrado em artifacts/preprocessed_v2/train. Execute a etapa de transformacao antes desta celula.")
sample_file = sample_files[0]
sample_engineered_df = pd.read_parquet(sample_file)

engineered_groups_df = pd.DataFrame(
    {
        "grupo": ["targets", "raw_targets", "raw_aux", "diff1", "dev_roll", "std_roll"],
        "quantidade": [
            len(bundle.target_columns),
            len(bundle.target_columns),
            len(bundle.auxiliary_columns),
            len(bundle.target_columns),
            len(bundle.target_columns),
            len(bundle.target_columns),
        ],
    }
)

display(sample_engineered_df.head())
engineered_groups_df


,series_id,well_name,well_id,file_path,timestamp,target__P-ANULAR,target__P-JUS-CKGL,target__P-MON-CKP,target__P-TPT,target__T-JUS-CKP,target__T-TPT,raw__P-ANULAR,raw__P-JUS-CKGL,raw__P-MON-CKP,raw__P-TPT,raw__T-JUS-CKP,raw__T-TPT,diff1__P-ANULAR,diff1__P-JUS-CKGL,diff1__P-MON-CKP,diff1__P-TPT,diff1__T-JUS-CKP,diff1__T-TPT,dev_roll5__P-ANULAR,dev_roll5__P-JUS-CKGL,dev_roll5__P-MON-CKP,dev_roll5__P-TPT,dev_roll5__T-JUS-CKP,dev_roll5__T-TPT,std_roll5__P-ANULAR,std_roll5__P-JUS-CKGL,std_roll5__P-MON-CKP,std_roll5__P-TPT,std_roll5__T-JUS-CKP,std_roll5__T-TPT
0,train__WELL-00001_20170201010207,WELL-00001,0,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,2017-02-01 01:02:07,1.997102,-1.728216,0.274243,0.495602,0.772344,0.714437,1.997102,-1.728216,0.274243,0.495602,0.772344,0.714437,0.086521,-0.501746,-0.000455,0.010641,0.003463,0.006845,0.087603,-1.426152,-0.000648,0.010976,0.003563,0.00701,-0.76087,-1.977433,-1.192783,-0.445059,-1.048903,-0.450833
1,train__WELL-00001_20170201010207,WELL-00001,0,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,2017-02-01 01:02:08,1.997102,-1.728216,0.306128,0.495602,0.756862,0.714437,1.997102,-1.728216,0.306128,0.495602,0.756862,0.714437,0.086521,-0.501746,0.249391,0.010641,-0.456846,0.006845,0.087603,-1.426152,0.066262,0.010976,-0.113900,0.00701,-0.76087,-1.977433,-1.009442,-0.445059,-0.745806,-0.450833
2,train__WELL-00001_20170201010207,WELL-00001,0,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,2017-02-01 01:02:09,1.997102,-1.728216,0.338006,0.495602,0.741399,0.714437,1.997102,-1.728216,0.338006,0.495602,0.741399,0.714437,0.086521,-0.501746,0.249346,0.010641,-0.456121,0.006845,0.087603,-1.426152,0.133156,0.010976,-0.231116,0.00701,-0.76087,-1.977433,-0.933523,-0.445059,-0.620596,-0.450833
3,train__WELL-00001_20170201010207,WELL-00001,0,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,2017-02-01 01:02:10,1.997102,-1.728216,0.369891,0.495602,0.725880,0.714437,1.997102,-1.728216,0.369891,0.495602,0.725880,0.714437,0.086521,-0.501746,0.249391,0.010641,-0.457571,0.006845,0.087603,-1.426152,0.200071,0.010976,-0.348919,0.00701,-0.76087,-1.977433,-0.858073,-0.445059,-0.495613,-0.450833
4,train__WELL-00001_20170201010207,WELL-00001,0,/home/tiagoriosrocha/Desktop/lstm-w3/3W/datase...,2017-02-01 01:02:11,1.997102,-1.728216,0.401770,0.495602,0.710399,0.714437,1.997102,-1.728216,0.401770,0.495602,0.710399,0.714437,0.086521,-0.501746,0.249346,0.010641,-0.456846,0.006845,0.087603,-1.426152,0.266964,0.010976,-0.466363,0.00701,-0.76087,-1.977433,-0.782856,-0.445059,-0.371157,-0.450833


,grupo,quantidade
0,targets,6
1,raw_targets,6
2,raw_aux,0
3,diff1,6
4,dev_roll,6
5,std_roll,6


## Versao 1 x Versao 2

A limpeza base deste notebook continua parecida com a da `versao1`: removemos o que nao ajuda, tratamos faltantes e preservamos o split temporal. A diferenca principal nao esta em "limpar de outro jeito", mas em **representar melhor o mesmo sinal para a rede**.

Na `versao1`, a entrada do modelo era essencialmente:

- as 6 variaveis finais ja limpas
- escaladas
- sem contexto adicional sobre velocidade, desvio local ou volatilidade

Na `versao2`, continuamos com essas 6 variaveis como alvo, mas criamos uma entrada mais rica:

- `raw__`: valor bruto padronizado
- `diff1__`: quanto o sinal mudou em relacao ao instante anterior
- `dev_roll5__`: quanto o valor atual se afasta da media curta
- `std_roll5__`: quao estavel ou instavel o sinal esta localmente
- colunas auxiliares adicionais, se elas realmente variam no treino

Em termos praticos, a `versao1` dizia ao modelo apenas "este e o valor atual". A `versao2` tenta dizer tambem:

- para onde o sinal esta indo
- o quanto ele esta acelerando ou freando
- o quanto ele esta diferente do comportamento local recente
- o quanto ele esta oscilando

Isso e importante porque, no seu problema, a persistencia ja e muito forte. Entao o modelo precisa enxergar mais do que nivel absoluto: ele precisa enxergar **dinamica**.


In [9]:
#monta uma comparacao didatica entre a entrada da versao1 e a entrada da versao2
comparison_overview_df = pd.DataFrame(
    {
        "aspecto": [
            "features usadas como alvo",
            "entrada da versao1",
            "entrada da versao2",
            "beneficio esperado da versao2",
        ],
        "descricao": [
            ", ".join(bundle.target_columns),
            "Somente os 6 sinais finais limpos e escalados.",
            "Sinais brutos padronizados + deltas + desvio local + volatilidade local + auxiliares validas.",
            "Dar ao modelo informacao sobre nivel, direcao, afastamento local e estabilidade do sinal.",
        ],
    }
)

example_timestamp = sample_engineered_df.loc[0, "timestamp"]
raw_reference_row = (
    train_df.loc[train_df["timestamp"] == example_timestamp, bundle.target_columns]
    .head(1)
    .T.reset_index()
)
raw_reference_row.columns = ["feature_versao1", "valor_limpo_no_timestamp"]

engineered_preview_columns = bundle.input_columns[: min(12, len(bundle.input_columns))]
engineered_reference_row = (
    sample_engineered_df.loc[sample_engineered_df["timestamp"] == example_timestamp, engineered_preview_columns]
    .head(1)
    .T.reset_index()
)
engineered_reference_row.columns = ["feature_versao2", "valor_processado_no_timestamp"]

print(f"Timestamp de exemplo: {example_timestamp}")
display(comparison_overview_df)
display(raw_reference_row)
display(engineered_reference_row)


Timestamp de exemplo: 2017-02-01 01:02:07


,aspecto,descricao
0,features usadas como alvo,"P-ANULAR, P-JUS-CKGL, P-MON-CKP, P-TPT, T-JUS-..."
1,entrada da versao1,Somente os 6 sinais finais limpos e escalados.
2,entrada da versao2,Sinais brutos padronizados + deltas + desvio l...
3,beneficio esperado da versao2,"Dar ao modelo informacao sobre nivel, direcao,..."


,feature_versao1,valor_limpo_no_timestamp
0,P-ANULAR,1.276773e+07
1,P-JUS-CKGL,1.563422e+06
2,P-MON-CKP,1.627884e+06
3,P-TPT,1.007454e+07
4,T-JUS-CKP,8.464463e+01
5,T-TPT,1.190781e+02


,feature_versao2,valor_processado_no_timestamp
0,raw__P-ANULAR,1.997102
1,raw__P-JUS-CKGL,-1.728216
2,raw__P-MON-CKP,0.274243
3,raw__P-TPT,0.495602
4,raw__T-JUS-CKP,0.772344
5,raw__T-TPT,0.714437
6,diff1__P-ANULAR,0.086521
7,diff1__P-JUS-CKGL,-0.501746
8,diff1__P-MON-CKP,-0.000455
9,diff1__P-TPT,0.010641


## 5. Fechamento

Nesta execucao da `versao2`, o pre-processamento produziu um resultado bem claro:

- o arquivo unico escolhido foi `WELL-00001_20170201010207.parquet`
- a serie tem `21.474` linhas e foi dividida em `15.031` para treino, `3.221` para validacao e `3.222` para teste
- nenhuma coluna auxiliar foi aproveitada, porque todas ficaram constantes no trecho de treino
- a entrada final do modelo ficou com `24` features:
  - `6` sinais `raw__`
  - `6` sinais `diff1__`
  - `6` sinais `dev_roll5__`
  - `6` sinais `std_roll5__`

Em outras palavras, a `versao2` nao melhorou por "achar mais variaveis". Ela melhorou porque passou a representar melhor a dinamica das mesmas 6 variaveis principais.
